# DepthwiseCNN — FFT-75 Benchmark

This notebook runs all three **DepthwiseCNN** variants on the two Kaggle inputs you uploaded:

- `FFT_75_512_1`   — 512-byte fragments
- `FFT_75_4096_1`  — 4096-byte fragments

Each dataset already contains `train.npz`, `val.npz`, and `test.npz`.

**Paper**: *File Fragment Type Classification Using Light-Weight Convolutional Neural Networks*

**Variants implemented:**
| Variant | Description |
|---------|-------------|
| `dsc`   | Depthwise Separable CNN (baseline) |
| `dsc_se`| DSC + Squeeze-and-Excitation channel attention |
| `m_dsc` | Multi-scale DSC (Inception-style, kernels 3/7/11) |

**Workflow:**
1. Install dependencies
2. Clone the repo
3. Locate and normalize the Kaggle inputs
4. Train and evaluate on 512B fragments
5. Train and evaluate on 4096B fragments
6. Paper comparison table + bundle all outputs

> **To switch fragment size:** change only `FRAGMENT_SIZE` in Cell 4.
> Everything else is automatic.


## Cell 1 — Install Dependencies

In [ ]:
import subprocess
import sys


def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])


pip('pyyaml>=6.0')
pip('scikit-learn>=1.5')
pip('pandas>=2.2')
pip('numpy>=1.26')
pip('matplotlib>=3.9')
pip('seaborn>=0.13')
pip('tqdm>=4.66')
pip('tabulate>=0.9')

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    print(f'VRAM    : {props.total_memory / 1e9:.1f} GB')


## Cell 2 — Clone Repo

In [ ]:
import gc
import os
import shutil
import sys
from pathlib import Path

WORKING = Path('/kaggle/working')
REPO_DIR = WORKING / 'deepcarv'
BRANCH_NAME = 'main'
GITHUB_TOKEN = ''  # Optional: set your token for private repos

if not REPO_DIR.exists() or not (REPO_DIR / 'src').exists():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    if GITHUB_TOKEN:
        clone_url = f'https://{GITHUB_TOKEN}@github.com/yuvnahr/deepcarv.git'
    else:
        clone_url = 'https://github.com/yuvnahr/deepcarv.git'
    os.system(f'git clone --depth 1 --branch {BRANCH_NAME} {clone_url} {REPO_DIR}')
else:
    print('Repo already present:', REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

os.environ['KAGGLE_RUNTIME'] = '1'

print('sys.path[0]:', sys.path[0])
print('src exists  :', (REPO_DIR / 'src').exists())
print('benchmark   :', (REPO_DIR / 'benchmarks' / 'DepthwiseCNN').exists())

gc.collect()


## Cell 3 — Locate and Normalize Kaggle Datasets

Reads the two mounted Kaggle inputs and normalizes them into the canonical layout:

- `/kaggle/working/data/FFT-75/512/{train,val,test}.npz`
- `/kaggle/working/data/FFT-75/4096/{train,val,test}.npz`

Handles both uppercase (`X`) and lowercase (`x`) NPZ keys transparently.


In [ ]:
import gc
from pathlib import Path

import numpy as np

from src.data.verify_dataset import verify_dataset

KAGGLE_INPUT   = Path('/kaggle/input')
CANONICAL_ROOT = WORKING / 'data' / 'FFT-75'
CANONICAL_ROOT.mkdir(parents=True, exist_ok=True)

# ---- Kaggle dataset slugs (match the names you uploaded) ------------------
DATASETS = {
    512:  KAGGLE_INPUT / 'FFT_75_512_1',
    4096: KAGGLE_INPUT / 'FFT_75_4096_1',
}
SPLITS = ('train', 'val', 'test')


def _load_npz_arrays(path: Path):
    """Load x/y arrays — handles both upper and lower case keys."""
    with np.load(path, allow_pickle=False) as data:
        keys = {k.lower(): k for k in data.files}
        if 'x' not in keys or 'y' not in keys:
            raise ValueError(f'{path.name} must contain x/y arrays, found {data.files}')
        return np.asarray(data[keys['x']]), np.asarray(data[keys['y']])


def _write_normalized_split(src: Path, dst: Path, fragment_size: int) -> None:
    X, y = _load_npz_arrays(src)
    if X.ndim == 1 and X.size % fragment_size == 0:
        X = X.reshape(-1, fragment_size)
    if X.ndim != 2 or X.shape[1] != fragment_size:
        raise ValueError(f'{src}: expected X shape [N, {fragment_size}], got {X.shape}')
    if y.ndim != 1:
        y = y.reshape(-1)
    if len(X) != len(y):
        raise ValueError(f'{src}: len(X)={len(X)} != len(y)={len(y)}')
    np.savez_compressed(dst, x=X, y=y)


def prepare_dataset(fragment_size: int, kaggle_root: Path) -> Path:
    if not kaggle_root.exists():
        raise FileNotFoundError(f'Missing Kaggle input folder: {kaggle_root}')

    target_dir = CANONICAL_ROOT / str(fragment_size)
    target_dir.mkdir(parents=True, exist_ok=True)

    print(f'\nPreparing {fragment_size}B dataset from: {kaggle_root}')
    for split in SPLITS:
        src = kaggle_root / f'{split}.npz'
        if not src.exists():
            raise FileNotFoundError(f'Missing split file: {src}')
        dst = target_dir / f'{split}.npz'
        _write_normalized_split(src, dst, fragment_size)
        X_tmp, y_tmp = _load_npz_arrays(dst)
        print(f'  {split}: {X_tmp.shape}  classes={len(set(y_tmp.tolist()))}')

    return target_dir


for frag_size, kaggle_root in DATASETS.items():
    prepare_dataset(frag_size, kaggle_root)

print('\n=== Canonical layout ===')
for frag_size in (512, 4096):
    for split in SPLITS:
        path = CANONICAL_ROOT / str(frag_size) / f'{split}.npz'
        print(f'  {frag_size}/{split}.npz -> {path.exists()}')

for frag_size in (512, 4096):
    print(f'\nVerifying {frag_size}B dataset...')
    verify_dataset(data_dir=CANONICAL_ROOT, fragment_size=frag_size)

gc.collect()
print('\nDataset preparation complete.')


## Cell 4 — Configuration

> **Change `FRAGMENT_SIZE` to `512` or `4096` to switch between scenarios.**
> Everything else adapts automatically.


In [ ]:
# ===========================================================================
# USER CONFIGURATION — change only this cell
# ===========================================================================

FRAGMENT_SIZE = 512   # 512 | 4096

# Variants to run in this session.
# Remove any variant you don't want to run.
VARIANTS_TO_RUN = ['dsc', 'dsc_se', 'm_dsc']

# Training hyperparameters
EPOCHS      = 50
BATCH_SIZE  = 256
LR          = 1e-3
SEED        = 42

# ===========================================================================
# Derived paths (do not edit)
# ===========================================================================
CKPT_DIR    = WORKING / 'checkpoints'
OUTPUTS_DIR = WORKING / 'outputs'
LOGS_DIR    = WORKING / 'logs'

for d in (CKPT_DIR, OUTPUTS_DIR, LOGS_DIR):
    d.mkdir(parents=True, exist_ok=True)

RESULTS = {}

print(f'Fragment size : {FRAGMENT_SIZE}B')
print(f'Variants      : {VARIANTS_TO_RUN}')
print(f'Epochs        : {EPOCHS}')
print(f'Batch size    : {BATCH_SIZE}')
print(f'Learning rate : {LR}')
print(f'Seed          : {SEED}')
print(f'Output dir    : {OUTPUTS_DIR}')


## Cell 5 — Train and Evaluate Helper

Defines `run_depthwisecnn()` which trains one variant and evaluates on the test split.
Called separately per variant in the cells below.


In [ ]:
import gc
import json

import torch

from benchmarks.DepthwiseCNN.scripts.train import main as dcnn_train_main
from benchmarks.DepthwiseCNN.scripts.evaluate import main as dcnn_eval_main


def run_depthwisecnn(variant: str, fragment_size: int,
                     epochs: int = EPOCHS,
                     batch_size: int = BATCH_SIZE,
                     lr: float = LR,
                     eval_batch_size: int = 1024) -> dict:
    """Train DepthwiseCNN <variant> on FFT-75 <fragment_size>B and evaluate on test split.

    Returns a result dict with accuracy, f1, and paths.
    """
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    run_name  = f'depthwisecnn_{variant}_fft75_{fragment_size}b'
    best_ckpt = CKPT_DIR / f'best_{run_name}.pt'
    run_dir   = OUTPUTS_DIR / run_name
    eval_dir  = OUTPUTS_DIR / f'{run_name}_eval'

    print(f'\n=== Training: {variant.upper()} | {fragment_size}B ===')

    # ---- Train ------------------------------------------------------------
    dcnn_train_main([
        '--data_dir',      str(CANONICAL_ROOT),
        '--fragment_size', str(fragment_size),
        '--variant',       variant,
        '--epochs',        str(epochs),
        '--batch_size',    str(batch_size),
        '--lr',            str(lr),
        '--seed',          str(SEED),
        '--run_dir',       str(run_dir),
        '--no_timestamp',
    ])

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    # The train script writes checkpoint_best.pt into run_dir.
    # Copy it to CKPT_DIR for the evaluate script.
    import shutil
    train_best = run_dir / 'checkpoint_best.pt'
    if train_best.exists():
        shutil.copy2(train_best, best_ckpt)

    # ---- Evaluate ---------------------------------------------------------
    print(f'\n=== Evaluating: {variant.upper()} | {fragment_size}B ===')

    dcnn_eval_main([
        '--checkpoint',    str(best_ckpt),
        '--data_dir',      str(CANONICAL_ROOT),
        '--fragment_size', str(fragment_size),
        '--variant',       variant,
        '--out_dir',       str(eval_dir),
        '--batch_size',    str(eval_batch_size),
    ])

    # ---- Collect metrics --------------------------------------------------
    metrics_file = eval_dir / 'metrics.json'
    metrics = {}
    if metrics_file.exists():
        with open(metrics_file, encoding='utf-8') as f:
            metrics = json.load(f)

    result = {
        'fragment_size': fragment_size,
        'variant':       variant,
        'accuracy':      metrics.get('accuracy', 0.0),
        'macro_f1':      metrics.get('macro_f1', 0.0),
        'weighted_f1':   metrics.get('weighted_f1', 0.0),
        'checkpoint':    str(best_ckpt),
        'run_dir':       str(run_dir),
        'eval_dir':      str(eval_dir),
    }

    key = f'{variant}_{fragment_size}'
    RESULTS[key] = result

    print(f'\n  Accuracy    : {result["accuracy"]:.4f}')
    print(f'  Macro F1    : {result["macro_f1"]:.4f}')
    print(f'  Weighted F1 : {result["weighted_f1"]:.4f}')

    return result


## Cell 6 — Run DSC Variant

In [ ]:
if 'dsc' in VARIANTS_TO_RUN:
    run_depthwisecnn(variant='dsc', fragment_size=FRAGMENT_SIZE)


## Cell 7 — Run DSC-SE Variant

In [ ]:
if 'dsc_se' in VARIANTS_TO_RUN:
    run_depthwisecnn(variant='dsc_se', fragment_size=FRAGMENT_SIZE)


## Cell 8 — Run M-DSC Variant

In [ ]:
if 'm_dsc' in VARIANTS_TO_RUN:
    run_depthwisecnn(variant='m_dsc', fragment_size=FRAGMENT_SIZE)


## Cell 9 — Results Comparison and Bundle

Prints a summary table across all variants and bundles every output file into a single `.zip`.


In [ ]:
import json
import shutil
from pathlib import Path

# ---- Save combined results JSON ------------------------------------------
results_path = OUTPUTS_DIR / f'depthwisecnn_fft75_{FRAGMENT_SIZE}b_results.json'
results_path.write_text(
    json.dumps(RESULTS, indent=2, sort_keys=True) + '\n',
    encoding='utf-8',
)

# ---- Print comparison table ----------------------------------------------
print(f'\n=== DepthwiseCNN Results — FFT-75 {FRAGMENT_SIZE}B ===')
print(f'{"Variant":<10}  {"Accuracy":>10}  {"Macro F1":>10}  {"Weighted F1":>12}')
print('-' * 48)
for variant in ('dsc', 'dsc_se', 'm_dsc'):
    key = f'{variant}_{FRAGMENT_SIZE}'
    r = RESULTS.get(key)
    if r:
        print(
            f'{variant:<10}  {r["accuracy"]:>10.4f}  '
            f'{r["macro_f1"]:>10.4f}  {r["weighted_f1"]:>12.4f}'
        )
    else:
        print(f'{variant:<10}  (not run)')

# ---- Bundle outputs ------------------------------------------------------
bundle_name = f'DepthwiseCNN_fft75_{FRAGMENT_SIZE}b_bundle'
bundle_dir  = WORKING / bundle_name
bundle_dir.mkdir(parents=True, exist_ok=True)

_EVAL_FILES = (
    'metrics.json',
    'summary.json',
    'predictions.csv',
    'confusion_matrix.csv',
    'per_class_metrics.csv',
    'classification_report.txt',
    'confusion_matrix.png',
)
_TRAIN_FILES = (
    'loss_curve.png',
    'accuracy_curve.png',
    'lr_curve.png',
)

for variant in ('dsc', 'dsc_se', 'm_dsc'):
    key = f'{variant}_{FRAGMENT_SIZE}'
    r = RESULTS.get(key)
    if not r:
        continue

    eval_dir = Path(r['eval_dir'])
    run_dir  = Path(r['run_dir'])
    ckpt     = Path(r['checkpoint'])
    prefix   = f'{FRAGMENT_SIZE}b_{variant}'

    for fname in _EVAL_FILES:
        src = eval_dir / fname
        if src.exists():
            shutil.copy2(src, bundle_dir / f'{prefix}_{fname}')

    for fname in _TRAIN_FILES:
        src = run_dir / fname
        if src.exists():
            shutil.copy2(src, bundle_dir / f'{prefix}_{fname}')

    if ckpt.exists():
        shutil.copy2(ckpt, bundle_dir / ckpt.name)

shutil.copy2(results_path, bundle_dir / results_path.name)

zip_path = WORKING / f'{bundle_name}.zip'
shutil.make_archive(str(zip_path.with_suffix('')), 'zip', bundle_dir)

print(f'\nBundle: {zip_path}')
print(f'Size  : {zip_path.stat().st_size / 1e6:.1f} MB')
print('Files :')
for f in sorted(bundle_dir.iterdir()):
    print(f'  {f.name}')
